##### Create SparkSession

In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
            .appName("Inflight Entertainment Analytics")
            .master("local[*]")
            .config("spark.sql.adaptive.enabled", "true")        # Performance optimization
            .config("spark.sql.shuffle.partitions", "100")       # Control parallelism
            .getOrCreate())

print("Spark version:", spark.version)


Spark version: 4.1.1


In [101]:
from pyspark.sql.functions import (
    col, count, sum, lit, log1p, explode, mean, min, max, round,
    split, to_timestamp, from_unixtime, year, month, 
    regexp_extract, trim, when, stddev
)

from pathlib import Path

##### Define paths for databases

In [42]:

bronze_db = Path("../database/bronze")
silver_db = Path("../database/silver")
gold_db   = Path("../database/gold")

dataset_path = Path("../dataset")
export_path = Path("../export")

##### Read CSV File

In [24]:
df_movies  = (
    spark
    .read
    .format("csv")
    .option("header", True)
    # .option("inferSchema", True)
    .load(str(dataset_path / "movies.csv"))
)

df_movies.printSchema()

df_movies.show(20, truncate=False)

root
 |-- movieId: string (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

+-------+-------------------------------------+-------------------------------------------+
|movieId|title                                |genres                                     |
+-------+-------------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                     |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                       |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)              |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)             |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)   |Comedy                                     |
|6      |Heat (1995)                          |Action|Crime|Thriller                      |
|7      |Sabrina (1995)                       |Comed

In [ ]:
# df_movies.summary().show(truncate=False)

##### Read Parquet File From Bronze Layer

In [11]:
df_movies = spark.read.format("parquet").load(str(bronze_db / "movies"))

df_movies.show(20, truncate=False)

+-------+-------------------------------------+-------------------------------------------+--------------------------+
|movieId|title                                |genres                                     |load_timestamp            |
+-------+-------------------------------------+-------------------------------------------+--------------------------+
|1      |Toy Story (1995)                     |Adventure|Animation|Children|Comedy|Fantasy|2026-06-02 16:48:40.323297|
|2      |Jumanji (1995)                       |Adventure|Children|Fantasy                 |2026-06-02 16:48:40.323297|
|3      |Grumpier Old Men (1995)              |Comedy|Romance                             |2026-06-02 16:48:40.323297|
|4      |Waiting to Exhale (1995)             |Comedy|Drama|Romance                       |2026-06-02 16:48:40.323297|
|5      |Father of the Bride Part II (1995)   |Comedy                                     |2026-06-02 16:48:40.323297|
|6      |Heat (1995)                          |A

In [15]:
df_ratings = spark.read.format("parquet").load(str(bronze_db / "ratings"))

df_ratings.printSchema()

df_ratings.show(20, truncate=False)

root
 |-- userId: string (nullable = true)
 |-- movieId: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)

+------+-------+------+---------+--------------------------+
|userId|movieId|rating|timestamp|load_timestamp            |
+------+-------+------+---------+--------------------------+
|1     |1      |4.0   |964982703|2026-06-02 16:48:41.802382|
|1     |3      |4.0   |964981247|2026-06-02 16:48:41.802382|
|1     |6      |4.0   |964982224|2026-06-02 16:48:41.802382|
|1     |47     |5.0   |964983815|2026-06-02 16:48:41.802382|
|1     |50     |5.0   |964982931|2026-06-02 16:48:41.802382|
|1     |70     |3.0   |964982400|2026-06-02 16:48:41.802382|
|1     |101    |5.0   |964980868|2026-06-02 16:48:41.802382|
|1     |110    |4.0   |964982176|2026-06-02 16:48:41.802382|
|1     |151    |5.0   |964984041|2026-06-02 16:48:41.802382|
|1     |157    |5.0   |964984100|2026-06-02 16:48:41.8023

##### Data Cleaning From Bronze Layer

In [14]:
# Regex pattern for checking and extracting the year at the end
year_pattern = r"\((\d{4})\)$"
full_pattern = r"(.*?)\s*\((\d{4})\)$"

df_movies_clean = (
    df_movies
    .withColumn("movieId", col("movieId").cast("int"))
    .withColumn("movieTitle", 
                when(
                    col("title").rlike(year_pattern), trim(regexp_extract("title", full_pattern, 1))
                ).otherwise(trim(col("title"))))
    .withColumn("year", 
                when(
                    col("title").rlike(year_pattern), regexp_extract("title", full_pattern, 2)
                ).otherwise(None).cast("int"))
    .withColumn("genres_list", split(col("genres"), "\\|"))
)

df_movies_clean.printSchema()

df_movies_clean.show(20, truncate=False)

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- movieTitle: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- genres_list: array (nullable = true)
 |    |-- element: string (containsNull = false)

+-------+-------------------------------------+-------------------------------------------+--------------------------+------------------------------+----+-------------------------------------------------+
|movieId|title                                |genres                                     |load_timestamp            |movieTitle                    |year|genres_list                                      |
+-------+-------------------------------------+-------------------------------------------+--------------------------+------------------------------+----+-------------------------------------------------+
|1      |Toy Story (1995)                    

In [16]:
df_ratings_clean = (
    df_ratings
    .orderBy("timestamp")
    .dropDuplicates(["userId", "movieId"])
    #.withColumn("datetime", to_timestamp(col("timestamp")))
    .withColumn("datetime", from_unixtime(col("timestamp").cast("long")))
    .withColumn("rating_year", year(col("datetime")))
    .withColumn("rating_month", month(col("datetime")))
    .withColumn("userId", col("userId").cast("int"))
    .withColumn("movieId", col("movieId").cast("int"))
    .withColumn("rating", col("rating").cast("float"))
)

df_ratings_clean.printSchema()

df_ratings_clean.show(20, truncate=False)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: float (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- datetime: string (nullable = true)
 |-- rating_year: integer (nullable = true)
 |-- rating_month: integer (nullable = true)

+------+-------+------+---------+--------------------------+-------------------+-----------+------------+
|userId|movieId|rating|timestamp|load_timestamp            |datetime           |rating_year|rating_month|
+------+-------+------+---------+--------------------------+-------------------+-----------+------------+
|1     |1      |4.0   |964982703|2026-06-02 16:48:41.802382|2000-07-31 02:45:03|2000       |7           |
|1     |1009   |3.0   |964981775|2026-06-02 16:48:41.802382|2000-07-31 02:29:35|2000       |7           |
|1     |1023   |5.0   |964982681|2026-06-02 16:48:41.802382|2000-07-31 02:44:41|2000       |7           |
|1     |1031   |5.0   |

##### Read Parquet File From Silver Layer

In [65]:
df_ratings = spark.read.format("parquet").load(str(silver_db / "ratings"))
df_movies = spark.read.format("parquet").load(str(silver_db / "movies"))
df_tags = spark.read.format("parquet").load(str(silver_db / "tags"))

In [30]:
df_ratings.show(20, truncate=False)

+------+-------+------+---------+--------------------------+-------------------+-----------+------------+
|userId|movieId|rating|timestamp|load_timestamp            |datetime           |rating_year|rating_month|
+------+-------+------+---------+--------------------------+-------------------+-----------+------------+
|1     |101    |5.0   |964980868|2026-06-02 16:48:41.802382|2000-07-31 02:14:28|2000       |7           |
|1     |1024   |5.0   |964982876|2026-06-02 16:48:41.802382|2000-07-31 02:47:56|2000       |7           |
|1     |1025   |5.0   |964982791|2026-06-02 16:48:41.802382|2000-07-31 02:46:31|2000       |7           |
|1     |1029   |5.0   |964982855|2026-06-02 16:48:41.802382|2000-07-31 02:47:35|2000       |7           |
|1     |1030   |3.0   |964982903|2026-06-02 16:48:41.802382|2000-07-31 02:48:23|2000       |7           |
|1     |1032   |5.0   |964982791|2026-06-02 16:48:41.802382|2000-07-31 02:46:31|2000       |7           |
|1     |1042   |4.0   |964981179|2026-06-02 16

In [34]:
df_ratings.summary().show(truncate=False)

+-------+------------------+------------------+------------------+--------------------+-------------------+------------------+------------------+
|summary|userId            |movieId           |rating            |timestamp           |datetime           |rating_year       |rating_month      |
+-------+------------------+------------------+------------------+--------------------+-------------------+------------------+------------------+
|count  |100836            |100836            |100836            |100836              |100836             |100836            |100836            |
|mean   |326.12756356856676|19435.2957177992  |3.501556983616962 |1.2059460873684695E9|NULL               |2007.7232535999049|6.419820302272997 |
|stddev |182.61849146350016|35530.987198700226|1.0425292390606302|2.1626103599513212E8|NULL               |6.890248941087953 |3.3991250557140718|
|min    |1                 |1                 |0.5               |1000129365          |1996-03-30 02:36:55|1996             

In [31]:
df_movies.show(20, truncate=False)

+-------+-------------------------------------+-------------------------------------------+--------------------------+------------------------------+----+-------------------------------------------------+
|movieId|title                                |genres                                     |load_timestamp            |movieTitle                    |year|genres_list                                      |
+-------+-------------------------------------+-------------------------------------------+--------------------------+------------------------------+----+-------------------------------------------------+
|1      |Toy Story (1995)                     |Adventure|Animation|Children|Comedy|Fantasy|2026-06-02 16:48:40.323297|Toy Story                     |1995|[Adventure, Animation, Children, Comedy, Fantasy]|
|2      |Jumanji (1995)                       |Adventure|Children|Fantasy                 |2026-06-02 16:48:40.323297|Jumanji                       |1995|[Adventure, Children, Fant

In [32]:
df_tags.show(20, truncate=False)

+------+-------+-----------------+----------+--------------------------+-------------------+---------+----------+
|userId|movieId|tag              |timestamp |load_timestamp            |datetime           |tags_year|tags_month|
+------+-------+-----------------+----------+--------------------------+-------------------+---------+----------+
|2     |60756  |funny            |1445714994|2026-06-02 16:48:45.810316|2015-10-25 03:29:54|2015     |10        |
|2     |60756  |Highly quotable  |1445714996|2026-06-02 16:48:45.810316|2015-10-25 03:29:56|2015     |10        |
|2     |60756  |will ferrell     |1445714992|2026-06-02 16:48:45.810316|2015-10-25 03:29:52|2015     |10        |
|2     |89774  |Boxing story     |1445715207|2026-06-02 16:48:45.810316|2015-10-25 03:33:27|2015     |10        |
|2     |89774  |MMA              |1445715200|2026-06-02 16:48:45.810316|2015-10-25 03:33:20|2015     |10        |
|2     |89774  |Tom Hardy        |1445715205|2026-06-02 16:48:45.810316|2015-10-25 03:33

##### Transformation to Gold Layer

In [66]:
df_movie_stats = (df_ratings.groupBy("movieId")
                .agg(
                    mean("rating").alias("avg_rating"),
                    count("*").alias("rating_count"),
                    stddev("rating").alias("rating_std")
                ))

df_movie_stats.show(truncate=False)

# df_movie_stats.orderBy(col("avg_rating").desc()).show(truncate=False)


+-------+------------------+------------+------------------+
|movieId|avg_rating        |rating_count|rating_std        |
+-------+------------------+------------+------------------+
|1580   |3.487878787878788 |165         |0.917232574091289 |
|1088   |3.369047619047619 |42          |0.994248851226354 |
|1037   |2.7708333333333335|24          |0.9438492725405695|
|537    |3.1666666666666665|12          |0.8348471099367218|
|5956   |3.518181818181818 |55          |0.9858425441648994|
|8641   |3.7719298245614037|57          |0.7323529559784174|
|3175   |3.58              |75          |1.0398804505092647|
|471    |3.55              |40          |1.1082441645256376|
|7380   |3.357142857142857 |7           |0.3779644730092272|
|100553 |4.5               |2           |0.0               |
|174551 |5.0               |1           |NULL              |
|307    |3.8958333333333335|24          |1.1513618275637831|
|32460  |4.25              |4           |0.6454972243679028|
|44022  |3.2173913043478

In [ ]:
# Save to CSV file
# df_movie_stats.toPandas().to_csv(str(export_path / "df_movie_stats.csv"), index=False)

In [68]:
df_tag_count = (df_tags.groupBy("movieId")
            .agg(count("*").alias("tag_count")))

df_tag_count.show(truncate=False)

# Save to CSV file
# df_tag_count.toPandas().to_csv(str(export_path / "df_tag_count.csv"), index=False)

+-------+---------+
|movieId|tag_count|
+-------+---------+
|8641   |7        |
|54997  |1        |
|307    |1        |
|326    |1        |
|471    |1        |
|1088   |2        |
|1096   |1        |
|1545   |1        |
|1580   |1        |
|1645   |1        |
|1959   |2        |
|2122   |1        |
|3175   |1        |
|3273   |2        |
|4361   |1        |
|6454   |1        |
|6466   |1        |
|6620   |1        |
|7833   |1        |
|8638   |1        |
+-------+---------+
only showing top 20 rows


In [69]:
df_movies_exploded = df_movies.withColumn("genre", explode(col("genres_list")))

df_movies_exploded.show(truncate=False)

# Save to CSV file
# df_movies_exploded.toPandas().to_csv(str(export_path / "df_movies_exploded.csv"), index=False)

+-------+----------------------------------+-------------------------------------------+--------------------------+---------------------------+----+-------------------------------------------------+---------+
|movieId|title                             |genres                                     |load_timestamp            |movieTitle                 |year|genres_list                                      |genre    |
+-------+----------------------------------+-------------------------------------------+--------------------------+---------------------------+----+-------------------------------------------------+---------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|2026-06-02 22:32:06.579272|Toy Story                  |1995|[Adventure, Animation, Children, Comedy, Fantasy]|Adventure|
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|2026-06-02 22:32:06.579272|Toy Story                  |1995|[Adventure, Anim

In [70]:
# Prepare movies data (only needed columns)
df_movies_clean = df_movies_exploded.select("movieId", "movieTitle", "genre")

df_movies_clean.show(truncate=False)

+-------+---------------------------+---------+
|movieId|movieTitle                 |genre    |
+-------+---------------------------+---------+
|1      |Toy Story                  |Adventure|
|1      |Toy Story                  |Animation|
|1      |Toy Story                  |Children |
|1      |Toy Story                  |Comedy   |
|1      |Toy Story                  |Fantasy  |
|2      |Jumanji                    |Adventure|
|2      |Jumanji                    |Children |
|2      |Jumanji                    |Fantasy  |
|3      |Grumpier Old Men           |Comedy   |
|3      |Grumpier Old Men           |Romance  |
|4      |Waiting to Exhale          |Comedy   |
|4      |Waiting to Exhale          |Drama    |
|4      |Waiting to Exhale          |Romance  |
|5      |Father of the Bride Part II|Comedy   |
|6      |Heat                       |Action   |
|6      |Heat                       |Crime    |
|6      |Heat                       |Thriller |
|7      |Sabrina                    |Com

In [71]:
# Join ratings with movie information
df_with_movies = df_ratings.join(df_movies_clean, "movieId")

df_with_movies.show(truncate=False)

+-------+------+------+---------+--------------------------+-------------------+-----------+------------+-----------------------+---------+
|movieId|userId|rating|timestamp|load_timestamp            |datetime           |rating_year|rating_month|movieTitle             |genre    |
+-------+------+------+---------+--------------------------+-------------------+-----------+------------+-----------------------+---------+
|101    |1     |5.0   |964980868|2026-06-02 22:32:07.997436|2000-07-31 02:14:28|2000       |7           |Bottle Rocket          |Romance  |
|101    |1     |5.0   |964980868|2026-06-02 22:32:07.997436|2000-07-31 02:14:28|2000       |7           |Bottle Rocket          |Crime    |
|101    |1     |5.0   |964980868|2026-06-02 22:32:07.997436|2000-07-31 02:14:28|2000       |7           |Bottle Rocket          |Comedy   |
|101    |1     |5.0   |964980868|2026-06-02 22:32:07.997436|2000-07-31 02:14:28|2000       |7           |Bottle Rocket          |Adventure|
|1024   |1     |5.0 

In [72]:
# Add movie statistics (avg_rating, rating_count, rating_std)
df_with_stats = df_with_movies.join(df_movie_stats, "movieId")

df_with_stats.show(truncate=False)

+-------+------+------+---------+--------------------------+-------------------+-----------+------------+-----------------------+---------+------------------+------------+------------------+
|movieId|userId|rating|timestamp|load_timestamp            |datetime           |rating_year|rating_month|movieTitle             |genre    |avg_rating        |rating_count|rating_std        |
+-------+------+------+---------+--------------------------+-------------------+-----------+------------+-----------------------+---------+------------------+------------+------------------+
|101    |1     |5.0   |964980868|2026-06-02 22:32:07.997436|2000-07-31 02:14:28|2000       |7           |Bottle Rocket          |Romance  |3.782608695652174 |23          |1.0425720702853738|
|101    |1     |5.0   |964980868|2026-06-02 22:32:07.997436|2000-07-31 02:14:28|2000       |7           |Bottle Rocket          |Crime    |3.782608695652174 |23          |1.0425720702853738|
|101    |1     |5.0   |964980868|2026-06-02 2

In [73]:
# Add tag counts (left join to keep movies without tags)
df_with_tags = df_with_stats.join(df_tag_count, "movieId", "left")

df_with_tags.show(truncate=False)

+-------+------+------+---------+--------------------------+-------------------+-----------+------------+-----------------------+---------+------------------+------------+------------------+---------+
|movieId|userId|rating|timestamp|load_timestamp            |datetime           |rating_year|rating_month|movieTitle             |genre    |avg_rating        |rating_count|rating_std        |tag_count|
+-------+------+------+---------+--------------------------+-------------------+-----------+------------+-----------------------+---------+------------------+------------+------------------+---------+
|101    |1     |5.0   |964980868|2026-06-02 22:32:07.997436|2000-07-31 02:14:28|2000       |7           |Bottle Rocket          |Romance  |3.782608695652174 |23          |1.0425720702853738|3        |
|101    |1     |5.0   |964980868|2026-06-02 22:32:07.997436|2000-07-31 02:14:28|2000       |7           |Bottle Rocket          |Crime    |3.782608695652174 |23          |1.0425720702853738|3     

In [74]:
# Replace null tag counts with 0
df_with_tags_filled = df_with_tags.fillna(0, subset=["tag_count"])

df_with_tags_filled.show(truncate=False)

+-------+------+------+---------+--------------------------+-------------------+-----------+------------+-----------------------+---------+------------------+------------+------------------+---------+
|movieId|userId|rating|timestamp|load_timestamp            |datetime           |rating_year|rating_month|movieTitle             |genre    |avg_rating        |rating_count|rating_std        |tag_count|
+-------+------+------+---------+--------------------------+-------------------+-----------+------------+-----------------------+---------+------------------+------------+------------------+---------+
|101    |1     |5.0   |964980868|2026-06-02 22:32:07.997436|2000-07-31 02:14:28|2000       |7           |Bottle Rocket          |Romance  |3.782608695652174 |23          |1.0425720702853738|3        |
|101    |1     |5.0   |964980868|2026-06-02 22:32:07.997436|2000-07-31 02:14:28|2000       |7           |Bottle Rocket          |Crime    |3.782608695652174 |23          |1.0425720702853738|3     

In [75]:
df_movie_genre_gold = (df_with_tags_filled
    .select(
        "movieId",
        "movieTitle",
        "genre",
        "rating",
        "rating_year",
        "rating_month",
        "avg_rating",
        "rating_count",
        "tag_count",
        "rating_std"
    )
    .orderBy(col("rating_count").desc(), col("avg_rating").desc())
)

df_with_tags_filled.show(truncate=False)

+-------+------+------+---------+--------------------------+-------------------+-----------+------------+-----------------------+---------+------------------+------------+------------------+---------+
|movieId|userId|rating|timestamp|load_timestamp            |datetime           |rating_year|rating_month|movieTitle             |genre    |avg_rating        |rating_count|rating_std        |tag_count|
+-------+------+------+---------+--------------------------+-------------------+-----------+------------+-----------------------+---------+------------------+------------+------------------+---------+
|101    |1     |5.0   |964980868|2026-06-02 22:32:07.997436|2000-07-31 02:14:28|2000       |7           |Bottle Rocket          |Romance  |3.782608695652174 |23          |1.0425720702853738|3        |
|101    |1     |5.0   |964980868|2026-06-02 22:32:07.997436|2000-07-31 02:14:28|2000       |7           |Bottle Rocket          |Crime    |3.782608695652174 |23          |1.0425720702853738|3     

In [79]:
df_movie_gold = (df_movie_genre_gold
            .groupBy("movieId", "movieTitle", "genre")
            .agg(
                mean("avg_rating").alias("avg_rating"),
                max("rating_count").alias("rating_count"),
                max("tag_count").alias("tag_count"),
                mean("rating_std").alias("rating_std")
            ))

df_movie_gold.show(truncate=False)

+-------+----------------------------------+---------+------------------+------------+---------+------------------+
|movieId|movieTitle                        |genre    |avg_rating        |rating_count|tag_count|rating_std        |
+-------+----------------------------------+---------+------------------+------------+---------+------------------+
|1      |Toy Story                         |Fantasy  |3.920930232558135 |215         |3        |0.8348591407114061|
|6874   |Kill Bill: Vol. 1                 |Thriller |3.9618320610687077|131         |2        |0.8426567889047473|
|5445   |Minority Report                   |Mystery  |3.637499999999993 |120         |1        |0.9004259029607522|
|185    |Net, The                          |Thriller |3.0401785714285716|112         |1        |1.0378824441260017|
|1917   |Armageddon                        |Thriller |3.054347826086959 |92          |1        |1.1055716082571212|
|2011   |Back to the Future Part II        |Adventure|3.5057471264367823

##### Read Parquet File from Gold Layer

In [108]:
df_movie_gold = spark.read.format("parquet").load(str(gold_db / "movies_gold"))
df_movie_genre_gold = spark.read.format("parquet").load(str(gold_db / "movie_genre_gold"))

df_movie_genre_gold.show(truncate=False)

+-------+---------------+--------+------+-----------+------------+------------------+------------+---------+------------------+
|movieId|movieTitle     |genre   |rating|rating_year|rating_month|avg_rating        |rating_count|tag_count|rating_std        |
+-------+---------------+--------+------+-----------+------------+------------------+------------+---------+------------------+
|2433   |Civil Action, A|Drama   |3.0   |2015       |6           |3.0454545454545454|11          |0        |0.650174801670499 |
|48877  |Saw III        |Thriller|2.5   |2011       |9           |3.0454545454545454|11          |0        |1.1927813180660036|
|48877  |Saw III        |Horror  |2.5   |2011       |9           |3.0454545454545454|11          |0        |1.1927813180660036|
|48877  |Saw III        |Crime   |2.5   |2011       |9           |3.0454545454545454|11          |0        |1.1927813180660036|
|48877  |Saw III        |Thriller|3.5   |2006       |11          |3.0454545454545454|11          |0     

In [109]:
min_value = df_movie_genre_gold.agg(min("rating_year")).collect()[0][0]

print(min_value)

1996


##### What are the top 10 genres by total number of ratings

In [114]:
grand_total = df_movie_genre_gold.count()

top_10_genres = (
    df_movie_genre_gold
        .groupBy("genre")
        .agg(count("*").alias("total_ratings"))
        .withColumn("percentage", round((col("total_ratings") / lit(grand_total) * 100), 2))
        .orderBy(col("total_ratings").desc())
        .limit(10)
)

print("=== Top 10 Genres by Total Number of Ratings ===")
top_10_genres.show(truncate=False)

=== Top 10 Genres by Total Number of Ratings ===
+---------+-------------+----------+
|genre    |total_ratings|percentage|
+---------+-------------+----------+
|Drama    |41928        |15.28     |
|Comedy   |39053        |14.23     |
|Action   |30635        |11.16     |
|Thriller |26452        |9.64      |
|Adventure|24161        |8.8       |
|Romance  |18124        |6.6       |
|Sci-Fi   |17243        |6.28      |
|Crime    |16681        |6.08      |
|Fantasy  |11834        |4.31      |
|Children |9208         |3.35      |
+---------+-------------+----------+



##### How has their popularity evolved over time (by year of rating)

In [115]:
# Popularity Evolution Over Time (Yearly Trend)
yearly_trend = (
    df_movie_genre_gold
        .groupBy("rating_year", "genre")
        .agg(count("*").alias("total_ratings"))
        .orderBy("rating_year", col("total_ratings").desc())
)

yearly_trend.show(truncate=False)

+-----------+-----------+-------------+
|rating_year|genre      |total_ratings|
+-----------+-----------+-------------+
|1996       |Drama      |2517         |
|1996       |Comedy     |2248         |
|1996       |Thriller   |1989         |
|1996       |Action     |1959         |
|1996       |Adventure  |1453         |
|1996       |Romance    |1288         |
|1996       |Crime      |1081         |
|1996       |Sci-Fi     |732          |
|1996       |Children   |699          |
|1996       |Fantasy    |552          |
|1996       |Animation  |357          |
|1996       |War        |336          |
|1996       |Musical    |317          |
|1996       |Mystery    |296          |
|1996       |Horror     |252          |
|1996       |Western    |250          |
|1996       |IMAX       |183          |
|1996       |Documentary|17           |
|1996       |Film-Noir  |9            |
|1997       |Drama      |799          |
+-----------+-----------+-------------+
only showing top 20 rows


In [121]:
# Filter to top 10 genres only
top_genres_list = [row['genre'] for row in top_10_genres.select("genre").collect()]

top_genres_trend = yearly_trend.filter(col("genre").isin(top_genres_list))

top_genres_trend.show(truncate=False)

+-----------+---------+-------------+
|rating_year|genre    |total_ratings|
+-----------+---------+-------------+
|1996       |Drama    |2517         |
|1996       |Comedy   |2248         |
|1996       |Thriller |1989         |
|1996       |Action   |1959         |
|1996       |Adventure|1453         |
|1996       |Romance  |1288         |
|1996       |Crime    |1081         |
|1996       |Sci-Fi   |732          |
|1996       |Children |699          |
|1996       |Fantasy  |552          |
|1997       |Drama    |799          |
|1997       |Comedy   |729          |
|1997       |Thriller |564          |
|1997       |Action   |541          |
|1997       |Adventure|447          |
|1997       |Romance  |401          |
|1997       |Crime    |273          |
|1997       |Sci-Fi   |264          |
|1997       |Fantasy  |176          |
|1997       |Children |166          |
+-----------+---------+-------------+
only showing top 20 rows


In [123]:
print("=== Genres Popularity over time by rating year ===")

# For single column
min_value = df_movie_genre_gold.agg(min("rating_year")).collect()[0][0]

(
top_genres_trend
    .filter(col("rating_year") >= min_value)
    .orderBy("rating_year", col("total_ratings").desc())
    .show(10, truncate=False)
)

=== Genres Popularity over time by rating year ===
+-----------+---------+-------------+
|rating_year|genre    |total_ratings|
+-----------+---------+-------------+
|1996       |Drama    |2517         |
|1996       |Comedy   |2248         |
|1996       |Thriller |1989         |
|1996       |Action   |1959         |
|1996       |Adventure|1453         |
|1996       |Romance  |1288         |
|1996       |Crime    |1081         |
|1996       |Sci-Fi   |732          |
|1996       |Children |699          |
|1996       |Fantasy  |552          |
+-----------+---------+-------------+
only showing top 10 rows


##### Identify "hidden gems": movies with fewer than 50 ratings but an average rating above 4.0.

In [124]:
# Identify Hidden Gems
hidden_gems = (
    df_movie_gold
        .filter((col("rating_count") < 50) & (col("avg_rating") > 4.0))
        .select("movieId", "movieTitle", "avg_rating", "rating_count")
        .orderBy(col("avg_rating").desc(), col("rating_count").desc())
)

print("\n=== Hidden Gems with Rating Count < 50 and Avg Rating > 4.0) ===")
hidden_gems.show(10, truncate=False)


=== Hidden Gems with Rating Count < 50 and Avg Rating > 4.0) ===
+-------+--------------------------------------------------------------------------+----------+------------+
|movieId|movieTitle                                                                |avg_rating|rating_count|
+-------+--------------------------------------------------------------------------+----------+------------+
|53     |Lamerica                                                                  |5.0       |2           |
|6442   |Belle époque                                                              |5.0       |2           |
|78836  |Enter the Void                                                            |5.0       |2           |
|6818   |Come and See (Idi i smotri)                                               |5.0       |2           |
|1151   |Lesson Faust                                                              |5.0       |2           |
|99     |Heidi Fleiss: Hollywood Madam                        

##### Which genres are over-represented in this segment compared to the overall genre distribution

In [126]:
# Overall genre distribution
overall_genre_dist = (
    df_movie_genre_gold
        .groupBy("genre")
        .agg(count("*").alias("overall_count"))
        .withColumn("overall_pct", round((col("overall_count") / lit(df_movie_genre_gold.count()) * 100), 2))
)

overall_genre_dist.orderBy("overall_pct", ascending=False).show(truncate=False)

+------------------+-------------+-----------+
|genre             |overall_count|overall_pct|
+------------------+-------------+-----------+
|Drama             |41928        |15.28      |
|Comedy            |39053        |14.23      |
|Action            |30635        |11.16      |
|Thriller          |26452        |9.64       |
|Adventure         |24161        |8.8        |
|Romance           |18124        |6.6        |
|Sci-Fi            |17243        |6.28       |
|Crime             |16681        |6.08       |
|Fantasy           |11834        |4.31       |
|Children          |9208         |3.35       |
|Mystery           |7674         |2.8        |
|Horror            |7291         |2.66       |
|Animation         |6988         |2.55       |
|War               |4859         |1.77       |
|Musical           |4138         |1.51       |
|IMAX              |4145         |1.51       |
|Western           |1930         |0.7        |
|Documentary       |1219         |0.44       |
|Film-Noir   

In [128]:
# Hidden gems genre distribution
gems_with_genre = hidden_gems.join(df_movie_genre_gold.select("movieId", "genre"), "movieId")

gems_genre_dist = (
    gems_with_genre
        .groupBy("genre")
        .agg(count("*").alias("gems_count"))
        .withColumn("gems_pct", round((col("gems_count") / lit(gems_with_genre.count()) * 100), 2))
)

gems_genre_dist.show(truncate=False)

+------------------+----------+--------+
|genre             |gems_count|gems_pct|
+------------------+----------+--------+
|Mystery           |517       |3.8     |
|Crime             |804       |5.9     |
|Romance           |952       |6.99    |
|Musical           |205       |1.5     |
|Thriller          |1054      |7.74    |
|Animation         |458       |3.36    |
|Adventure         |735       |5.4     |
|Film-Noir         |237       |1.74    |
|(no genres listed)|14        |0.1     |
|Drama             |3737      |27.43   |
|IMAX              |32        |0.23    |
|War               |643       |4.72    |
|Documentary       |300       |2.2     |
|Horror            |232       |1.7     |
|Western           |287       |2.11    |
|Fantasy           |362       |2.66    |
|Comedy            |1589      |11.66   |
|Children          |224       |1.64    |
|Action            |816       |5.99    |
|Sci-Fi            |424       |3.11    |
+------------------+----------+--------+



In [129]:
# Compare both distributions
comparison = overall_genre_dist.join(gems_genre_dist, "genre", "left").fillna(0)

comparison.show(truncate=False)

+------------------+-------------+-----------+----------+--------+
|genre             |overall_count|overall_pct|gems_count|gems_pct|
+------------------+-------------+-----------+----------+--------+
|Mystery           |7674         |2.8        |517       |3.8     |
|Crime             |16681        |6.08       |804       |5.9     |
|Romance           |18124        |6.6        |952       |6.99    |
|Musical           |4138         |1.51       |205       |1.5     |
|Thriller          |26452        |9.64       |1054      |7.74    |
|Animation         |6988         |2.55       |458       |3.36    |
|Adventure         |24161        |8.8        |735       |5.4     |
|Film-Noir         |870          |0.32       |237       |1.74    |
|(no genres listed)|47           |0.02       |14        |0.1     |
|Drama             |41928        |15.28      |3737      |27.43   |
|IMAX              |4145         |1.51       |32        |0.23    |
|War               |4859         |1.77       |643       |4.72 

In [133]:
# Calculate difference distribution
comparison = (
    comparison
        .withColumn("over_represented_diff", round(col("gems_pct") - col("overall_pct"), 2))
        .orderBy(col("over_represented_diff").desc())
)

print("\n=== Genres over-represented in hidden gems vs overall ===")
(
comparison
    .select(
        "genre", 
        "overall_pct", 
        "gems_pct", 
        "over_represented_diff"
    )
    .show(truncate=False)
)


=== Genres over-represented in hidden gems vs overall ===
+------------------+-----------+--------+---------------------+
|genre             |overall_pct|gems_pct|over_represented_diff|
+------------------+-----------+--------+---------------------+
|Drama             |15.28      |27.43   |12.15                |
|War               |1.77       |4.72    |2.95                 |
|Documentary       |0.44       |2.2     |1.76                 |
|Film-Noir         |0.32       |1.74    |1.42                 |
|Western           |0.7        |2.11    |1.41                 |
|Mystery           |2.8        |3.8     |1.0                  |
|Animation         |2.55       |3.36    |0.81                 |
|Romance           |6.6        |6.99    |0.39                 |
|(no genres listed)|0.02       |0.1     |0.08                 |
|Musical           |1.51       |1.5     |-0.01                |
|Crime             |6.08       |5.9     |-0.18                |
|Horror            |2.66       |1.7     |-0.9

##### Simulate an "inflight catalog" constraint: if you could only offer 100 movies on a flight, <br> which 100 would you pick to maximize expected passenger satisfaction across diverse tastes? <br> Describe your selection criteria (e.g., genre coverage, rating quality, popularity balance) and implement it.

In [150]:
# Criteria 1 = Filter movies with minimum credibility
popularity = df_movie_gold.filter(col("rating_count") >= 10)

popularity.show(truncate=False)


+-------+--------------------------------------------------------+------------------+------------+---------+------------------+
|movieId|movieTitle                                              |avg_rating        |rating_count|tag_count|rating_std        |
+-------+--------------------------------------------------------+------------------+------------+---------+------------------+
|1136   |Monty Python and the Holy Grail                         |4.161764705882377 |136         |4        |0.9065244025257879|
|48774  |Children of Men                                         |3.9453125         |64          |0        |0.8550373030941566|
|71535  |Zombieland                                              |3.8773584905660434|53          |6        |0.8765407334942278|
|783    |Hunchback of Notre Dame, The                            |3.329787234042555 |47          |0        |0.9282791216329179|
|3082   |World Is Not Enough, The                                |3.3333333333333295|39          |0     

In [138]:
n = 100

# Criteria 2 = Balance average rating and popularity by calculating Bayesian Score
catalog = popularity.withColumn(
    "score", 
    col("avg_rating") * log1p(col("rating_count"))
)

# Select Top N by score
final_catalog = catalog.orderBy(col("score").desc()).limit(n)

final_catalog.show(truncate=False)

+-------+-----------------------------------------------------------------------+------------------+------------+---------+------------------+------------------+
|movieId|movieTitle                                                             |avg_rating        |rating_count|tag_count|rating_std        |score             |
+-------+-----------------------------------------------------------------------+------------------+------------+---------+------------------+------------------+
|318    |Shawshank Redemption, The                                              |4.429022082018944 |317         |4        |0.7130186250284273|25.520252812061198|
|356    |Forrest Gump                                                           |4.164133738601782 |329         |9        |0.8312435184333592|24.14819737571684 |
|296    |Pulp Fiction                                                           |4.197068403908814 |307         |181      |0.9519971466349181|24.049620750363143|
|2571   |Matrix, The        

In [139]:
# Criteria 3 = Ensure Genre Coverage
catalog_with_genres = final_catalog.join(
    df_movie_genre_gold.select("movieId", "genre"), 
    "movieId"
)

catalog_with_genres.show(truncate=False)

+-------+------------+-----------------+------------+---------+------------------+-----------------+-------+
|movieId|movieTitle  |avg_rating       |rating_count|tag_count|rating_std        |score            |genre  |
+-------+------------+-----------------+------------+---------+------------------+-----------------+-------+
|356    |Forrest Gump|4.164133738601782|329         |9        |0.8312435184333592|24.14819737571684|War    |
|356    |Forrest Gump|4.164133738601782|329         |9        |0.8312435184333592|24.14819737571684|Romance|
|356    |Forrest Gump|4.164133738601782|329         |9        |0.8312435184333592|24.14819737571684|Drama  |
|356    |Forrest Gump|4.164133738601782|329         |9        |0.8312435184333592|24.14819737571684|Comedy |
|356    |Forrest Gump|4.164133738601782|329         |9        |0.8312435184333592|24.14819737571684|War    |
|356    |Forrest Gump|4.164133738601782|329         |9        |0.8312435184333592|24.14819737571684|Romance|
|356    |Forrest Gu

In [145]:
genre_coverage = (catalog_with_genres
    .groupBy("movieTitle","genre")
    .agg(count("*").alias("movie_count"))
    .orderBy(col("movie_count").desc()))

print("\nGenre Coverage in Catalog:")
genre_coverage.show(truncate=False)


Genre Coverage in Catalog:
+----------------------------------+---------+-----------+
|movieTitle                        |genre    |movie_count|
+----------------------------------+---------+-----------+
|Forrest Gump                      |Drama    |329        |
|Forrest Gump                      |Romance  |329        |
|Forrest Gump                      |War      |329        |
|Forrest Gump                      |Comedy   |329        |
|Shawshank Redemption, The         |Drama    |317        |
|Shawshank Redemption, The         |Crime    |317        |
|Pulp Fiction                      |Crime    |307        |
|Pulp Fiction                      |Drama    |307        |
|Pulp Fiction                      |Comedy   |307        |
|Pulp Fiction                      |Thriller |307        |
|Silence of the Lambs, The         |Thriller |279        |
|Silence of the Lambs, The         |Horror   |279        |
|Silence of the Lambs, The         |Crime    |279        |
|Matrix, The                

In [149]:
print(f"\n=== Inflight Catalog ===")

# Top N Recommended Movies
print(f"Top {n} Recommended Movies for Inflight:")

(
final_catalog
    .join(genre_coverage, "movieTitle")
    .select (
        "movieTitle", 
        round(col("avg_rating"), 2).alias("avg_rating"), 
        "rating_count", 
        round(col("score"), 2).alias("score")
    )
    .distinct()
    .orderBy(col("score").desc())
    .show(n, truncate=False)
)


=== Inflight Catalog ===
Top 100 Recommended Movies for Inflight:
+-----------------------------------------------------------------------+----------+------------+-----+
|movieTitle                                                             |avg_rating|rating_count|score|
+-----------------------------------------------------------------------+----------+------------+-----+
|Shawshank Redemption, The                                              |4.43      |317         |25.52|
|Forrest Gump                                                           |4.16      |329         |24.15|
|Pulp Fiction                                                           |4.2       |307         |24.05|
|Matrix, The                                                            |4.19      |278         |23.61|
|Silence of the Lambs, The                                              |4.16      |279         |23.45|
|Star Wars: Episode IV - A New Hope                                     |4.23      |251         |23.4